# State · Node · Edge — LangGraph 필수 구성요소

LangGraph 의 그래프는 세 가지로 이루어진다.

| 구성요소 | 한 줄 정의 |
|---|---|
| **State (상태)** | Agent 의 현재 상태를 나타내는 데이터 구조 |
| **Node (노드)** | Agent 가 수행하는 논리를 구현하는 함수 |
| **Edge (엣지)** | 한 노드에서 다음 노드로의 연결(흐름) |

여기에 **Reducer (리듀서)** — 노드의 업데이트가 State 에 적용되는 방식 — 까지 더하면 기초가 완성된다.

> 이 노트북은 LLM/API 키 없이 동작한다. 그래프 메커니즘 자체에 집중한다.

## State (상태)

State 는 Agent 의 현재 상태를 나타내는 데이터 구조이며, 주로 `TypedDict` 또는 Pydantic `BaseModel` 로 정의한다.

### 1. `TypedDict` 와 Pydantic `BaseModel` 이해하기

> #### 📎 사전 지식: `TypedDict` / Pydantic `BaseModel` 이 뭔가?
>
> 둘 다 **"딕셔너리(dict)에 정해진 모양을 부여하는" 도구**다. 파이썬 `dict` 는 아무 키나 아무 타입이나 넣을 수 있어 자유롭지만, "이 데이터는 반드시 `id`(정수)와 `name`(문자)을 가져야 한다" 같은 약속을 강제하지 못한다. 그 약속을 미리 선언하는 것이 이 둘이다.
>
> | | `TypedDict` | Pydantic `BaseModel` |
> |---|---|---|
> | 출신 | **파이썬 표준** (`typing`, 설치 불필요) | **외부 라이브러리** (`pydantic`) |
> | 실체 | 그냥 dict | 클래스 인스턴스(객체) |
> | 값 접근 | `u["name"]` | `u.name` |
> | 잘못된 타입을 넣으면? | **그냥 통과** (IDE/타입검사기만 경고) | **런타임에 `ValidationError` 로 막음** |
> | 무게 | 매우 가벼움 | 약간 무거움 |
>
> → 빠른 프로토타이핑이면 `TypedDict`, 외부 입력 검증이 중요하면 Pydantic.
>
> **`class User(TypedDict):` 의 소괄호는?** → **상속**을 뜻한다(생성자 파라미터가 아니다). 파이썬에서 `class 이름(부모):` 의 괄호는 "어떤 부모 클래스를 물려받는가"를 적는 자리다. 정의할 때 괄호(상속)와, 인스턴스를 만들 때 괄호(`User(id=1, name="alice")`, 값 전달)는 의미가 다르다.

In [ ]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    email: str

user1: User = {
    'id': 1,
    'name': 'alice',
    'email': 'alice@example.com',
}
print(user1)

`TypedDict` 는 런타임 검증을 하지 않는다. 아래처럼 `name` 에 숫자를 넣어도 오류 없이 그대로 통과한다.

In [ ]:
# name 자리에 잘못된 타입(int)을 넣어도 그대로 실행됨
user2: User = {
    'id': 1,
    'name': 123,
    'email': 'alice@example.com',
}
print(user2)

반면 Pydantic `BaseModel` 은 선언한 타입을 실제로 검증한다.

In [ ]:
from pydantic import BaseModel

class UserModel(BaseModel):
    id: int
    name: str
    email: str

user_data = {
    'id': 1,
    'name': 'alice',
    'email': 'alice@example.com',
}
user1 = UserModel(**user_data)
print(user1)

- 잘못된 데이터 타입이면 오류 메시지를 반환한다.

In [ ]:
from pydantic import ValidationError

bad_data = {
    'id': 1,
    'name': 123,   # str 이 아님
    'email': 'alice@example.com',
}
try:
    UserModel(**bad_data)
except ValidationError as e:
    print(e)

### 2. 상태 - Graph Schema 지정하기

State 는 **그래프의 스키마**와, 상태에 업데이트를 적용하는 방법을 지정하는 **리듀서 함수**로 구성된다.

- 에이전트를 구현할 때, 에이전트가 가지고 있어야 할 정보를 상태를 통해 업데이트한다.
- 아래 예시는 사용자의 질문과 답변을 정해진 데이터 구조(State)에 맞춰 입력받고 반환한다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# 입력을 위한 스키마 정의
class InputState(TypedDict):
    question: str

# 출력을 위한 스키마 정의
class OutputState(TypedDict):
    answer: str

# 입력과 출력을 합한 종합 스키마 정의
class OverallState(InputState, OutputState):
    pass

In [ ]:
# 입력을 처리하고 답변을 생성하는 노드 정의
def answer_node(state: InputState):
    return {"answer": "bye", "question": state["question"]}  # 상태 업데이트

graph_builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
graph_builder.add_node(answer_node)              # 답변 노드 추가
graph_builder.add_edge(START, "answer_node")     # 시작 엣지 추가
graph_builder.add_edge("answer_node", END)       # 끝 엣지 추가
graph = graph_builder.compile()                  # 그래프 컴파일

# 입력 invoke 및 결과 출력 (output 스키마인 answer 만 반환)
print(graph.invoke({"question": "hi"}))

컴파일된 그래프 구조는 mermaid 텍스트로 확인할 수 있다.

In [ ]:
print(graph.get_graph().draw_mermaid())

### 3. 상태 - Reducer 지정하기

리듀서(Reducer)는 노드의 업데이트가 State 에 적용되는 방식을 지정하는 함수다.

**1) 리듀서가 지정되지 않은 경우** — 새 값으로 덮어쓴다.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    value1: int
    value2: list[str]

**2) 리듀서가 지정된 경우 (`operator.add`)** — 기존 값에 새 값을 더한다(리스트는 이어붙임).

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    value1: int
    value2: Annotated[list[str], add]

**3) 리듀서가 지정된 경우 (`custom add`)** — 병합 함수를 직접 정의할 수도 있다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

def add(left, right):
    """operator 의 add 를 직접 구현해본 형태"""
    return left + right

class State(TypedDict):
    value1: int
    value2: Annotated[list[str], add]

**4) 리듀서가 지정된 경우 (`add_messages`)**

`add_messages` 는 기존 메시지에 새 메시지를 병합하는 리듀서다. 메시지 누적에 특화되어 있어, 같은 `id` 의 메시지는 교체하고 새 메시지는 이어붙인다.

```python
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]
add_messages(msgs1, msgs2)
```

In [ ]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

## Nodes (노드)

노드는 Agent 가 수행하는 논리를 구현하는 함수로 표현된다.

### 1. 노드를 추가하는 방법

`add_node("이름", 함수)` 로 등록한다.

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(dict)

def my_node(state: dict):
    return {"results": f"Hello, {state['input']}!"}

def my_other_node(state: dict):
    return state

builder.add_node("my_node", my_node)
builder.add_node("other_node", my_other_node)

### 2. 노드의 기능을 구현하는 방법

노드 함수는 현재 State 를 입력받아, 변경할 부분을 dict 로 반환한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from operator import add

class State(TypedDict):
    messages: Annotated[list[str], add]

graph = StateGraph(State)

In [ ]:
def chatbot(state: State):
    answer = "안녕하세요! 무엇을 도와드릴까요?"
    print("Answer : ", answer)
    return {"messages": [answer]}

graph.add_node("chatbot", chatbot)

In [ ]:
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
graph = graph.compile()

In [ ]:
graph.invoke({"messages": ["안녕!"]})

## Edges (엣지)

**기본 엣지 (Normal Edges)**: 한 노드에서 다음 노드로 바로 이동한다.

```python
graph.add_edge("node_a", "node_b")
```

조건에 따라 다음 노드를 고르는 **조건부 엣지(conditional edges)** 는 이후 노트북에서 다룬다.

## 정리

- **State** = 그래프의 데이터 스키마 (`TypedDict` / Pydantic `BaseModel`)
- **Reducer** = 업데이트 병합 규칙. 없으면 덮어쓰기, `operator.add` 는 누적, `add_messages` 는 메시지 전용
- **Node** = State 를 받아 변경분 dict 를 반환하는 함수
- **Edge** = 노드 간 흐름 (`START` → … → `END`)

다음: 메시지 상태를 업데이트하고 실행 결과를 받는 다양한 방법을 다룬다.